<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "388"
iso_country = "Jamaica"
year = 2024

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    "TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Jamaica': 

Number of trade records present in dataset: 24765
Number of partner countries present in dataset: 116
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$6858.2m
Total weight within the dataset: 6845.2 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,TandT_match_desc,TandT_ad_val_pct,TandT_N,TandT_ciffob,TandT_cifvalue,TandT_fobvalue,cost_int_match_desc,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,388,Jamaica,276,Germany,Import,Sea,220890,"220890 | Spirits, liqueurs an",2208,"2208 | Ethyl alcohol, unden",22,"22 | Beverages, spirits a",2413.0,Undenatured ethyl alcohol of an alcoholic stre...,Food & Beverages,26.0,B_T - Beverages and tobacco products,6854.60,NaN,6854.60,l,2322.0,2322.0,1083.712260,6530.0,4057.552630,4397.219531,"['Container', 'Chemical tanker']","['Container', 'Chemical tanker']","1 - Reporter, Partner and HS4 code",4.79,1,313.33,6854.60,6541.27,1. Primary Int. Inv. with Vess Type,15.568859,15.531179,15.534704,15.534704,15.554305,15.554305,16.297762,29.121174,17.742975,16.559334,16.615276,17.099154,15.952614,28.298233,18.792616,17.247850,16.855513,17.637938,15.649983,31.384326,22.858353,20.421934,18.936004,18.382427,15.352569,33.316038,26.937184,25.162505,20.053986,19.703414,15.060727,33.376920,32.436144,32.489088,21.658855,20.684093,14.763105,33.105293,31.571370,34.119597,26.04316,21.052869,14.482807,29.073400,30.820008,33.073262,24.887277,21.293166,14.197241,26.539420,26.984528,32.246085,26.600109,21.443829,2413
1,1,388,Jamaica,124,Canada,Import,Sea,220429,"220429 | Wine; still, in cont",2204,2204 | Wine of fresh grapes,22,"22 | Beverages, spirits a",24212.0,"Wine of fresh grapes, except sparkling wine; g...",Food & Beverages,26.0,B_T - Beverages and tobacco products,548.02,NaN,548.02,l,18.5,205.5,160.624662,4011.0,2492.319081,400.327910,['Container'],['Container'],"1 - Reporter, Partner and HS4 code",12.31,1,60.07,548.02,487.95,1. Primary Int. Inv. with Vess Type,1.372704,1.368993,1.369338,1.369338,1.371302,1.371302,1.441082,2.646714,1.578973,1.471222,1.476449,1.516801,1.408698,2.573796,1.668375,1.530497,1.492735,1.567561,1.380305,2.848677,2.065276,1.820776,1.679716,1.637672,1.352395,3.002950,2.424240,2.258822,1.786545,1.761655,1.325007,3.050322,2.962635,2.943695,1.938527,1.853917,1.297077,3.017117,2.862245,3.122208,2.35932,1.888698,1.270775,2.635934,2.815291,3.003581,2.234244,1.911332,1.243983,2.395233,2.433570,2.930772,2.410196,1.925451,24212


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [5]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $1600.1m (23.3%), ... 

                              cifvalue   vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                             
21 | Miscellaneous edible  228284228.5   54994277.7     14.3      3.0
10 | Cereals               203544420.7  647281842.2     12.7     35.5
22 | Beverages, spirits a  161714175.0  238680791.7     10.1     13.1
20 | Preparations of vege  146658907.3   96207931.1      9.2      5.3
19 | Preparations of cere  127476276.6   44849304.2      8.0      2.5
04 | Dairy produce; birds  112945691.2   46079971.3      7.1      2.5
17 | Sugars and sugar con  110180390.6  294776716.4      6.9     16.1
02 | Meat and edible meat   94332678.3   33508290.9      5.9      1.8
23 | Food industries, res   87463118.2  175366183.2      5.5      9.6
16 | Meat, fish, crustace   86308461.3   17003564.9      5.4      0.9
03 | Fish and crustaceans   74258903.7   21774794.2      4

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [14]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $1379.2m (20.1%), ... 

                     cifvalue   vol_kg_vtw  pct_val  pct_vol
eora_desc                                                   
Food & Beverages  913373634.6  877772914.6     13.3     12.8
Agriculture       439383728.2  645856014.4      6.4      9.4
Fishing            26407278.7    3566511.6      0.4      0.1



### 2.1.3 GTAP (16x headings)

In [7]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $1247.4m (18.2%), ... 

                                  cifvalue   vol_kg_vtw  pct_val  pct_vol
gtap_desc                                                                
OFD - Food products n.e.c.     649584353.6  279597478.4      9.5      4.1
VOL - Vegetable oils and fats  124742006.4  197306581.8      1.8      2.9
SGR - Sugar                     90130840.0  285026124.4      1.3      4.2
GRO - Cereal grains n.e.c.      80249865.9  331941683.0      1.2      4.8
CMT - Bovine meat prods         72436860.4   22585671.7      1.1      0.3
MIL - Dairy products            68164733.7   17753630.2      1.0      0.3
PCR - Processed rice            58997254.8  102478231.6      0.9      1.5
OMT - Meat products n.e.c.      44998409.4   16019006.5      0.7      0.2
V_F - Vegetables, fruit, nuts   34657715.5   46206640.0      0.5      0.7
PDR - Paddy rice                10456597.0   18163108.9      0.2      0.3
OCR - Crops n.e.c.               9793725.0   

## 2.2 Comparison of results

In [8]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 23.3% 
GTAP 18.2% 
EORA 20.1%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [9]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [10]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_21385/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,TandT_match_desc,TandT_ad_val_pct,TandT_N,TandT_ciffob,TandT_cifvalue,TandT_fobvalue,cost_int_match_desc,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,388,Jamaica,276,Germany,Import,Sea,220890,"220890 | Spirits, liqueurs an",2208,"2208 | Ethyl alcohol, unden",22,"22 | Beverages, spirits a",2413.0,Undenatured ethyl alcohol of an alcoholic stre...,Food & Beverages,26.0,B_T - Beverages and tobacco products,6854.60,NaN,6854.60,l,2322.0,2322.0,1083.712260,6530.0,4057.552630,4397.219531,"['Container', 'Chemical tanker']","['Container', 'Chemical tanker']","1 - Reporter, Partner and HS4 code",4.79,1,313.33,6854.60,6541.27,1. Primary Int. Inv. with Vess Type,15.568859,15.531179,15.534704,15.534704,15.554305,15.554305,16.297762,29.121174,17.742975,16.559334,16.615276,17.099154,15.952614,28.298233,18.792616,17.247850,16.855513,17.637938,15.649983,31.384326,22.858353,20.421934,18.936004,18.382427,15.352569,33.316038,26.937184,25.162505,20.053986,19.703414,15.060727,33.376920,32.436144,32.489088,21.658855,20.684093,14.763105,33.105293,31.571370,34.119597,26.04316,21.052869,14.482807,29.073400,30.820008,33.073262,24.887277,21.293166,14.197241,26.539420,26.984528,32.246085,26.600109,21.443829,2413,NaN
1,1,388,Jamaica,124,Canada,Import,Sea,220429,"220429 | Wine; still, in cont",2204,2204 | Wine of fresh grapes,22,"22 | Beverages, spirits a",24212.0,"Wine of fresh grapes, except sparkling wine; g...",Food & Beverages,26.0,B_T - Beverages and tobacco products,548.02,NaN,548.02,l,18.5,205.5,160.624662,4011.0,2492.319081,400.327910,['Container'],['Container'],"1 - Reporter, Partner and HS4 code",12.31,1,60.07,548.02,487.95,1. Primary Int. Inv. with Vess Type,1.372704,1.368993,1.369338,1.369338,1.371302,1.371302,1.441082,2.646714,1.578973,1.471222,1.476449,1.516801,1.408698,2.573796,1.668375,1.530497,1.492735,1.567561,1.380305,2.848677,2.065276,1.820776,1.679716,1.637672,1.352395,3.002950,2.424240,2.258822,1.786545,1.761655,1.325007,3.050322,2.962635,2.943695,1.938527,1.853917,1.297077,3.017117,2.862245,3.122208,2.35932,1.888698,1.270775,2.635934,2.815291,3.003581,2.234244,1.911332,1.243983,2.395233,2.433570,2.930772,2.410196,1.925451,24212,NaN


## 3.3 Inspect the output

In [11]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $930.2m of $6858.2m (13.6%)
Volume: 1229.3 of 6845.2 million kilograms (18.0%)
Value-to-Weight: 0.76 from 1.0 US$ per kg (+-24.5%)
Average distance to market (volume-weighted): 2345.8 from 2750.9 nm (+-14.7%)

            food_group  N_com     cifvalue   vol_kg_vtw  pct_val  pct_vol
0    Processed Cereals    479  436426671.9  242544072.9      6.4      3.5
1  Unprocessed Cereals     18  134090569.0  526640501.7      2.0      7.7
2      Sugar and Honey     84  110487084.8  294864645.7      1.6      4.3
3           Vegetables    265   70051010.9   76468706.0      1.0      1.1
4                Dairy     87   62569616.6   17860605.6      0.9      0.3
5        Fats and Oils    102   60244623.0   49783386.3      0.9      0.7
6        Meat and Prep     40   44832654.6   14623509.5      0.7      0.2
7               Fruits    120   11501768.8    6483379.2      0.2      0.1



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [16]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,26183818.9,47312496.3,28571707.0,26662453.6,26754196.6,27505889.3,24148834.5,54437552.1,52825124.6,52741958.7,34945860.9,33418875.3,22725799.4,43027734.3,43747923.3,52415967.8,43168386.2,34684718.2
1,Fats and Oils,63982539.3,107327645.2,68667776.8,64327414.2,64520041.1,66654728.8,59778792.0,121431348.1,118137004.3,120656076.7,82748632.9,78766629.9,56844367.8,99469780.2,101319699.5,118849203.5,98763948.5,81327139.2
2,Fruits,7209543.2,13235357.6,7898487.1,7359963.5,7386084.0,7587972.4,6629363.5,15253902.9,14814220.2,14721364.2,9696158.8,9272970.8,6224380.2,11980032.2,12171858.3,14656445.7,12052931.7,9630607.5
3,Meat and Prep,17357916.6,31879827.4,19018822.7,17720955.8,17783913.3,18269960.1,15959785.2,36741322.0,35685117.0,35456987.3,23349678.3,22330546.2,14983848.9,28850727.1,29312502.9,35301328.9,29030953.6,23192180.3
4,Processed Cereals,250854363.3,449565528.0,273039508.1,255137603.5,256024716.4,263267123.3,231691262.7,520125490.0,503190707.5,503769501.3,334647723.3,318783591.8,218312673.4,411021962.8,417898379.5,498918551.6,411548480.5,330655537.5
5,Sugar and Honey,369571773.7,608488312.1,394893182.4,370508043.2,371583985.3,384230788.8,346359891.0,688575362.9,669439336.3,687458937.6,475428029.8,450906778.7,330157218.5,568115519.8,578879398.6,674724006.2,562345744.9,464930810.8
6,Unprocessed Cereals,247711330.1,434176664.9,268031789.0,251085866.8,251933099.6,259317743.5,229699267.3,506136481.2,489250731.3,491894953.2,330446182.6,311212711.7,217126315.3,402170516.1,408812062.9,483892681.0,401375267.7,322033325.0
7,Vegetables,90743082.6,164921481.0,99142259.9,92491226.5,92817204.2,95390258.4,83596713.1,190294033.4,184451070.4,183922805.1,121519938.1,116132117.2,78607195.7,149834215.9,152295694.5,182744886.5,150428140.6,120563720.2


### 3.3.2 Total FEM costs by scenario (% cif)

In [15]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.34,0.04,0.01,0.01,0.02,0.48,0.46,0.46,0.17,0.15,0.32,0.34,0.47,0.33,0.19
1,Fats and Oils,0.72,0.08,0.01,0.01,0.04,1.02,0.97,1.01,0.38,0.32,0.71,0.74,1.03,0.70,0.41
2,Fruits,0.52,0.06,0.01,0.02,0.03,0.75,0.71,0.70,0.27,0.23,0.50,0.52,0.73,0.51,0.30
3,Meat and Prep,0.32,0.04,0.01,0.01,0.02,0.46,0.44,0.43,0.16,0.14,0.31,0.32,0.45,0.31,0.18
4,Processed Cereals,0.46,0.05,0.01,0.01,0.03,0.66,0.62,0.62,0.24,0.20,0.44,0.46,0.64,0.44,0.26
5,Sugar and Honey,2.16,0.23,0.01,0.02,0.13,3.10,2.92,3.09,1.17,0.95,2.15,2.25,3.12,2.10,1.22
6,Unprocessed Cereals,1.39,0.15,0.03,0.03,0.09,2.06,1.94,1.96,0.75,0.61,1.38,1.43,1.99,1.37,0.78
7,Vegetables,1.06,0.12,0.02,0.03,0.07,1.52,1.44,1.43,0.54,0.46,1.02,1.05,1.49,1.03,0.60
